In [6]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from docling_core.transforms.chunker import HierarchicalChunker
from langchain_core.documents import Document


# Chunking and Embeding

## Extraction

In [ ]:
## Note :- all manual logic fails and required different logic for different pdf ,current useful method is using
## Docling devolped by the IBM

In [ ]:
## first run the following command for the extraction of the table of content
## pdftotext -f 9 -l 16 -layout ECBC_2024.pdf toc_raw.txt

In [7]:
converter = DocumentConverter()

In [ ]:
## Basic PDF Extraction
result = converter.convert("D:\\AiEnergyAssitanceForBuilding\\PdfResourc\\Used\\ECBC_HVAC_TIP_SHEET_2011.pdf")

In [9]:
document = result.document
markdown_output = document.export_to_markdown()
json_output = document.export_to_dict()

In [10]:
print(len(markdown_output))

72869


## Chunking using langchain

In [11]:
chunker = HybridChunker(max_tokens=500,merge_peers=True)

In [14]:
chunk_iter  = chunker.chunk(dl_doc=result.document)
chunks = list(chunk_iter)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (577 > 512). Running this sequence through the model will result in indexing errors


In [15]:
len(chunks)

60

In [ ]:
chunks[1]

## 1  Structure-Aware Chunking + semantic using docling

                    Docling
                       │
                       ▼
              Structure extraction
                       │
              ┌────────┴────────┐
              │                 │
          Section             Table
              │                 │
              ▼                 ▼
       Paragraph grouping    Table chunk
              │
              ▼
       Size/token check
              │
       ┌──────┴──────┐
       │             │
    Small          Large
       │             │
       ▼             ▼
     Keep       Paragraph/
                sentence split

In [24]:
def structure_aware_chunking(doc):
    """
    Create structure-aware chunks from a DoclingDocument.

    The chunking logic:
    - Keeps headings associated with their content.
    - Preserves document/page metadata where available.
    - Treats tables separately.
    - Creates a new chunk when a new heading/section is encountered.
    """

    chunks = []

    current_heading = []
    current_content = []
    current_page = None

    def save_chunk():
        if not current_content:
            return

        content = "\n".join(current_content).strip()

        if not content:
            return

        metadata = {
            "heading": " > ".join(current_heading),
            "page": current_page,
            "content_type": "text"
        }

        chunks.append(
            Document(
                page_content=content,
                metadata=metadata
            )
        )

    # Iterate through Docling document items
    for item, level in doc.iterate_items():

        item_type = item.__class__.__name__

        # -------------------------
        # Heading
        # -------------------------
        if item_type in ["SectionHeaderItem", "TitleItem"]:

            # Save previous section
            save_chunk()

            current_content.clear()

            text = getattr(item, "text", "").strip()

            if text:
                # Maintain heading hierarchy
                if level is not None:

                    current_heading = current_heading[:level]

                    current_heading.append(text)

                else:
                    current_heading.append(text)

            continue

        # -------------------------
        # Table
        # -------------------------
        if item_type == "TableItem":

            # Save text before table
            save_chunk()
            current_content.clear()

            try:
                table_text = item.export_to_markdown(doc)

            except Exception:
                table_text = str(item)

            chunks.append(
                Document(
                    page_content=table_text,
                    metadata={
                        "heading": " > ".join(current_heading),
                        "page": current_page,
                        "content_type": "table"
                    }
                )
            )

            continue

        # -------------------------
        # Normal text
        # -------------------------
        text = getattr(item, "text", None)

        if text:

            text = text.strip()

            if text:
                current_content.append(text)

        # -------------------------
        # Page information
        # -------------------------
        prov = getattr(item, "prov", None)

        if prov:
            try:
                current_page = prov[0].page_no
            except Exception:
                pass

    # Save final chunk
    save_chunk()

    return chunks


In [26]:
structure_Aware_cunks = structure_aware_chunking(document)

In [27]:
len(structure_Aware_cunks)

61

In [ ]:
for i, chunk in enumerate(structure_Aware_cunks[:5]):

    print("\n" + "=" * 80)
    print(f"CHUNK {i + 1}")

    print("\nMetadata:")
    print(chunk.metadata)

    print("\nContent:")
    print(chunk.page_content[:1000])

## Hierarchial structure

In [25]:
def get_page_number(item):
    prov = getattr(item, "prov", None)

    if prov:
        try:
            return prov[0].page_no
        except Exception:
            pass

    return None

In [26]:
def create_parent_child_chunks(doc):

    parents = []

    current_parent = None
    current_parent_content = []

    parent_counter = 0
    child_counter = 0

    # -----------------------------------------------------
    # Save current parent
    # -----------------------------------------------------

    def save_parent():

        nonlocal current_parent
        nonlocal current_parent_content

        if current_parent is None:
            return

        current_parent["content"] = current_parent_content.copy()

        parents.append(current_parent)

        current_parent = None
        current_parent_content = []


    # -----------------------------------------------------
    # Iterate through Docling document
    # -----------------------------------------------------

    for item, level in doc.iterate_items():

        item_type = item.__class__.__name__

        page_number = get_page_number(item)

        # =================================================
        # HEADING
        # =================================================

        if item_type in [
            "SectionHeaderItem",
            "TitleItem"
        ]:

            heading_text = getattr(item, "text", "")

            if not heading_text:
                continue

            heading_text = heading_text.strip()

            # New parent section
            save_parent()

            parent_counter += 1

            current_parent = {

                "parent_id": f"parent_{parent_counter}",

                "title": heading_text,

                "level": level,

                "page": page_number,

                "content": []

            }

            continue


        # =================================================
        # TABLE
        # =================================================

        if item_type == "TableItem":

            # If no heading exists yet, create a default parent
            if current_parent is None:

                parent_counter += 1

                current_parent = {

                    "parent_id": f"parent_{parent_counter}",

                    "title": "Document Content",

                    "level": 0,

                    "page": page_number,

                    "content": []

                }

            try:

                table_markdown = item.export_to_markdown(doc)

            except Exception:

                table_markdown = str(item)


            current_parent_content.append({

                "type": "table",

                "content": table_markdown,

                "page": page_number

            })

            continue


        # =================================================
        # TEXT
        # =================================================

        text = getattr(item, "text", None)

        if text:

            text = text.strip()

            if text:

                # If document starts with text before heading
                if current_parent is None:

                    parent_counter += 1

                    current_parent = {

                        "parent_id": f"parent_{parent_counter}",

                        "title": "Document Introduction",

                        "level": 0,

                        "page": page_number,

                        "content": []

                    }


                current_parent_content.append({

                    "type": "text",

                    "content": text,

                    "page": page_number

                })


    # Save final parent
    save_parent()

    # -----------------------------------------------------
    # Create child LangChain Documents
    # -----------------------------------------------------

    child_documents = []

    for parent in parents:

        for element in parent["content"]:

            child_counter += 1

            # =============================================
            # TEXT CHILD
            # =============================================

            if element["type"] == "text":

                child_documents.append(

                    Document(

                        page_content=element["content"],

                        metadata={

                            "chunk_type": "child",

                            "content_type": "text",

                            "child_id": f"child_{child_counter}",

                            "parent_id": parent["parent_id"],

                            "parent_title": parent["title"],

                            "parent_level": parent["level"],

                            "page": element["page"]

                        }

                    )

                )


            # =============================================
            # TABLE CHILD
            # =============================================

            elif element["type"] == "table":

                child_documents.append(

                    Document(

                        page_content=element["content"],

                        metadata={

                            "chunk_type": "child",

                            "content_type": "table",

                            "child_id": f"child_{child_counter}",

                            "parent_id": parent["parent_id"],

                            "parent_title": parent["title"],

                            "parent_level": parent["level"],

                            "page": element["page"]

                        }

                    )

                )


    return parents, child_documents



In [27]:
hirarchial_chunking = create_parent_child_chunks(document)

In [28]:
hirarchial_chunking

([{'parent_id': 'parent_1',
   'title': 'EnErgy COnSErVATIOn BUILDIng CODE TIP SHEET',
   'level': 1,
   'page': 1,
   'content': [{'type': 'text',
     'content': 'Version 2.0 - March, 2011',
     'page': 1}]},
  {'parent_id': 'parent_2',
   'title': 'HVAC SySTEM',
   'level': 1,
   'page': 1,
   'content': [{'type': 'text',
     'content': "Credits: E Source Technology Atlas US DOE's Building Technologies Program",
     'page': 1},
    {'type': 'text', 'content': 'Alliance to Save Energy', 'page': 1},
    {'type': 'text', 'content': 'USAID ECO-III Project', 'page': 1},
    {'type': 'text', 'content': 'International Resources Group', 'page': 1},
    {'type': 'text', 'content': 'Phone: +91-11-4597 4597', 'page': 1},
    {'type': 'text', 'content': 'Fax: +91-11-2685 3114', 'page': 1},
    {'type': 'text', 'content': 'Email: eco3@irgssa.com', 'page': 1},
    {'type': 'text',
     'content': "Heating, Ventilation and Air Conditioning (HVAC) accounts for a significant portion of a commerci

In [17]:
chunker = HierarchicalChunker()
chunks = list(chunker.chunk(document))

In [23]:
chunks[87]

DocChunk(text='HVAC equipment are required to meet or exceed  minimum efficiency requirements mentioned in ECBC 5.2.2. Power consumption ratings for unitary air conditioners,  split  systems  and  packaged air conditioners are referred to BIS codes (See  Table  2,  3  and  4).  Cooling  systems not  included  are  referred  to  ASHRAE 90.1  -  2004.  Single  zone  unitary  systems are  covered  as  well  as  multiple  zone  air and water systems. The more complex the system,  the  more  requirements  apply  to that  system:  a  single-zone  unitary  system has  fewer  requirements  than  a  complex system  made  up  of  chillers,  boilers,  and fan coil units.', meta=DocMeta(schema_name='docling_core.transforms.chunker.DocMeta', version='1.0.0', doc_items=[TextItem(self_ref='#/texts/215', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TEXT: 'text'>, prov=[ProvenanceItem(page_no=5, bbox=BoundingBox(l=393.5956, t=303.

In [19]:
len(chunks)

221